# 17.07 - Saving and Loading Models

**Notebook type:** Practice notebook with theory, exercises, TODO cells, and test cases.

**Daily output:** A save/load reproducibility test: train one epoch, save a versioned checkpoint, create a fresh model and optimizer, reload the checkpoint, and reproduce the original predictions.

A reliable checkpoint is more than a weight file. Today you will preserve the model state, optimizer state, epoch, architecture config, class mapping, and human-readable notes needed to restart inference or training safely.

## Core Ideas

### 1. Save state, not the live model object

`state_dict()` stores named tensors for a model or optimizer. Saving state dictionaries is easier to inspect, migrate, and load across code changes than pickling an entire live model. Recreate the architecture from recorded config, then call `load_state_dict(...)`.

### 2. A useful checkpoint is a small experiment record

Weights alone do not explain how to rebuild a run. Record the epoch, model dimensions, learning rate, ordered class names, checkpoint format version, and concise notes. For a real project, also preserve preprocessing, validation score, git revision, and random-number-generator states when exact training continuation matters.

### 3. Name files predictably

Names such as `day17_tiny_classifier_epoch_001.pt` sort naturally and make the training point explicit. Avoid repeatedly overwriting an ambiguous `model.pt`; keep a separate clearly chosen `best.pt` only when the selection rule is documented.

### 4. Load onto an explicit device

Use `map_location` so a GPU-produced checkpoint can be inspected on CPU. Recreate the same architecture before loading. For inference, call `eval()` and disable gradients. For training continuation, restore optimizer and scheduler state as well.

### 5. Verify behavior, not just file existence

A successful `torch.load` does not prove the correct architecture, class order, or preprocessing was used. Capture reference logits before saving, load into fresh objects, rerun the same inputs, and assert that logits and predictions match within an explicit tolerance.

In [ ]:
import os
import numpy as np
import torch
from torch import nn

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cpu")
CLASS_NAMES = ["red", "green", "blue"]
CONFIG = {
    "input_dim": 4,
    "hidden_dim": 8,
    "num_classes": 3,
    "learning_rate": 0.20,
    "seed": SEED,
}
CHECKPOINT_PATH = os.path.join(
    "_day17_checkpoints", "day17_tiny_classifier_epoch_001.pt"
)


## Prepared Toy Data and Model

The deterministic feature vectors form three compact classes. The provided generator and model keep the exercises focused on checkpoint safety rather than data plumbing.

**Return structure — `make_day17_data(...)`:** a 2-tuple `(features, labels)` where `features` is a CPU `torch.Tensor` with dtype `torch.float32` and shape `[N, 4]`, and `labels` is a CPU `torch.Tensor` with dtype `torch.long` and shape `[N]`. `N` equals `samples_per_class * 3`.

**Return structure — `TinyCheckpointClassifier(...)`:** a `TinyCheckpointClassifier` module. Calling it as `model(features)` invokes `forward` and returns a floating-point logits `torch.Tensor` on the input device with shape `[N, 3]`.

In [ ]:
def make_day17_data(samples_per_class=12):
    generator = torch.Generator().manual_seed(SEED)
    centers = torch.tensor(
        [[1.0, 0.0, 0.0, 0.5], [0.0, 1.0, 0.0, -0.5], [0.0, 0.0, 1.0, 0.25]],
        dtype=torch.float32,
    )
    features = []
    labels = []
    for class_index, center in enumerate(centers):
        noise = 0.08 * torch.randn(samples_per_class, 4, generator=generator)
        features.append(center.unsqueeze(0) + noise)
        labels.append(torch.full((samples_per_class,), class_index, dtype=torch.long))
    return torch.cat(features, dim=0), torch.cat(labels, dim=0)


class TinyCheckpointClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, features):
        return self.network(features)


train_features, train_labels = make_day17_data()
model = TinyCheckpointClassifier(
    CONFIG["input_dim"], CONFIG["hidden_dim"], CONFIG["num_classes"]
).to(DEVICE)
optimizer = torch.optim.SGD(model.parameters(), lr=CONFIG["learning_rate"])
train_features = train_features.to(DEVICE)
train_labels = train_labels.to(DEVICE)
print("features:", tuple(train_features.shape), train_features.dtype, train_features.device)
print("labels:", tuple(train_labels.shape), train_labels.dtype, train_labels.device)
print("class mapping:", dict(enumerate(CLASS_NAMES)))


## Exercise 17-A: Train One Epoch

Implement one deterministic full-batch optimization step. Clear old gradients, compute cross-entropy, backpropagate, and update the parameters.

**Return structure — `train_one_epoch(...)`:** a dictionary with exactly two keys:

- `loss`: Python `float`, the cross-entropy value computed before the parameter update
- `predictions`: detached CPU `torch.Tensor`, dtype `torch.long`, shape `[N]`, containing the pre-update argmax class indices

In [ ]:
def train_one_epoch(model, features, labels, optimizer):
    # TODO 17-A: use train mode, clear gradients, compute cross-entropy,
    # backpropagate, update once, and return the documented dictionary.
    raise NotImplementedError("Implement train_one_epoch")


# Smoke check: train exactly one epoch on the prepared batch.
train_report = train_one_epoch(model, train_features, train_labels, optimizer)
print("epoch-1 loss:", round(train_report["loss"], 4))
print("prediction shape:", tuple(train_report["predictions"].shape))


## Exercise 17-B: Capture Deterministic Reference Logits

Implement inference that temporarily switches to evaluation mode, disables gradients, and restores the model's original mode. These logits are the behavioral reference for the save/load test.

**Return structure — `predict_logits(...)`:** one detached CPU floating-point `torch.Tensor` with shape `[N, C]`, where `N` is the number of feature rows and `C` is the number of classes. The result has no gradient history.

In [ ]:
def predict_logits(model, features):
    # TODO 17-B: run deterministic inference with eval mode and no gradients,
    # restore the original model mode, and return detached CPU logits.
    raise NotImplementedError("Implement predict_logits")


# Smoke check: capture the reference output that loading must reproduce.
reference_logits = predict_logits(model, train_features)
reference_predictions = reference_logits.argmax(dim=1)
print("reference logits:", tuple(reference_logits.shape), reference_logits.dtype)
print("reference predictions:", reference_predictions[:8].tolist())


## Exercise 17-C: Save a Complete Checkpoint

Create the parent folder and save a dictionary with the exact keys `checkpoint_version`, `epoch`, `model_state`, `optimizer_state`, `config`, `class_names`, and `notes`. Keep the class-name list ordered because output column `i` must retain the same meaning after loading.

**Return structure — `save_checkpoint(...)`:** one Python `str` equal to `checkpoint_path`. The function also writes a non-empty PyTorch checkpoint file at that path and returns only after the write succeeds.

In [ ]:
def save_checkpoint(checkpoint_path, model, optimizer, epoch, config, class_names, notes):
    # TODO 17-C: create the parent directory, assemble every documented key,
    # save with torch.save, and return the path.
    raise NotImplementedError("Implement save_checkpoint")


# Smoke check: save a named, inspectable epoch checkpoint.
saved_path = save_checkpoint(
    CHECKPOINT_PATH,
    model,
    optimizer,
    epoch=1,
    config=CONFIG,
    class_names=CLASS_NAMES,
    notes="One full-batch training step on deterministic Day 17 toy data.",
)
print("saved checkpoint:", saved_path)
print("checkpoint bytes:", os.path.getsize(saved_path))


## Exercise 17-D: Load into Fresh Objects

Load with an explicit `map_location` and `weights_only=True`, validate the checkpoint schema, restore the model, and optionally restore the optimizer. Return metadata rather than duplicating the large state dictionaries.

**Return structure — `load_checkpoint(...)`:** a dictionary with the exact schema:

- `checkpoint_version`: Python `int`
- `epoch`: Python `int`
- `config`: `dict[str, int | float]` with keys `input_dim`, `hidden_dim`, `num_classes`, `learning_rate`, and `seed`
- `class_names`: `list[str]` of length `C`, in model-output order
- `notes`: Python `str`
- `optimizer_loaded`: Python `bool`, true exactly when an optimizer argument was supplied and restored
- `checkpoint_path`: Python `str` equal to the loaded path

The supplied model is moved to `device` and mutated in place; the optional optimizer is also mutated in place. No model or optimizer object is returned.

In [ ]:
def load_checkpoint(checkpoint_path, model, optimizer=None, device=torch.device("cpu")):
    # TODO 17-D: load safely onto device, validate required keys, restore model
    # and optional optimizer state, then return the documented metadata only.
    raise NotImplementedError("Implement load_checkpoint")


# Smoke check: simulate a restarted notebook with fresh objects, then restore.
restored_model = TinyCheckpointClassifier(
    CONFIG["input_dim"], CONFIG["hidden_dim"], CONFIG["num_classes"]
).to(DEVICE)
restored_optimizer = torch.optim.SGD(restored_model.parameters(), lr=CONFIG["learning_rate"])
load_report = load_checkpoint(CHECKPOINT_PATH, restored_model, restored_optimizer, DEVICE)
restored_logits = predict_logits(restored_model, train_features)
print("loaded epoch:", load_report["epoch"])
print("predictions identical:", torch.equal(reference_predictions, restored_logits.argmax(dim=1)))
print("max logit difference:", float((reference_logits - restored_logits).abs().max()))


## Reproducibility Interpretation

Exact equality is expected here because both inference passes use the same CPU inputs, architecture, state tensors, and evaluation behavior. On different devices or library builds, tiny floating-point differences can occur; compare logits with a documented tolerance and require identical predicted classes. If outputs differ materially, check architecture config, preprocessing, model mode, checkpoint selection, and class order before blaming numerical precision.

## Test Cases

Run this cell after completing all TODO cells. The tests verify data contracts, one-epoch training output, checkpoint naming and schema, model and optimizer restoration, class mapping, mode restoration, and exact prediction reproducibility. A correct implementation prints `Day 17 tests passed`.

**Return structure — `run_day17_tests()`:** `None`. Success is communicated by completed assertions and the printed confirmation.

In [ ]:
def run_day17_tests():
    assert train_features.shape == (36, CONFIG["input_dim"])
    assert train_features.dtype == torch.float32
    assert train_features.device.type == "cpu"
    assert train_labels.shape == (36,)
    assert train_labels.dtype == torch.long
    assert set(train_labels.tolist()) == {0, 1, 2}

    assert set(train_report) == {"loss", "predictions"}
    assert isinstance(train_report["loss"], float)
    assert train_report["loss"] > 0.0
    assert train_report["predictions"].shape == train_labels.shape
    assert train_report["predictions"].dtype == torch.long
    assert train_report["predictions"].device.type == "cpu"

    model.train()
    checked_logits = predict_logits(model, train_features)
    assert model.training, "predict_logits must restore the original model mode"
    assert checked_logits.shape == (36, len(CLASS_NAMES))
    assert checked_logits.dtype == torch.float32
    assert checked_logits.device.type == "cpu"
    assert not checked_logits.requires_grad

    assert saved_path == CHECKPOINT_PATH
    assert os.path.isfile(CHECKPOINT_PATH)
    assert os.path.getsize(CHECKPOINT_PATH) > 0
    assert os.path.basename(CHECKPOINT_PATH) == "day17_tiny_classifier_epoch_001.pt"

    raw = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
    assert set(raw) == {
        "checkpoint_version",
        "epoch",
        "model_state",
        "optimizer_state",
        "config",
        "class_names",
        "notes",
    }
    assert raw["checkpoint_version"] == 1
    assert raw["epoch"] == 1
    assert raw["config"] == CONFIG
    assert raw["class_names"] == CLASS_NAMES

    fresh_model = TinyCheckpointClassifier(
        CONFIG["input_dim"], CONFIG["hidden_dim"], CONFIG["num_classes"]
    ).to(DEVICE)
    fresh_optimizer = torch.optim.SGD(
        fresh_model.parameters(), lr=CONFIG["learning_rate"]
    )
    metadata = load_checkpoint(CHECKPOINT_PATH, fresh_model, fresh_optimizer, DEVICE)
    fresh_logits = predict_logits(fresh_model, train_features)
    assert set(metadata) == {
        "checkpoint_version",
        "epoch",
        "config",
        "class_names",
        "notes",
        "optimizer_loaded",
        "checkpoint_path",
    }
    assert metadata["optimizer_loaded"] is True
    assert metadata["class_names"] == CLASS_NAMES
    assert fresh_optimizer.param_groups[0]["lr"] == CONFIG["learning_rate"]
    assert torch.equal(fresh_logits, reference_logits)
    assert torch.equal(fresh_logits.argmax(dim=1), reference_predictions)
    print("Day 17 tests passed")


run_day17_tests()


## Day 17 Checklist

- [ ] I save model and optimizer `state_dict` values instead of pickling a live model.
- [ ] I record the epoch, architecture config, learning rate, class order, format version, and notes.
- [ ] I use explicit, sortable checkpoint names such as `epoch_001`.
- [ ] I recreate the architecture before calling `load_state_dict`.
- [ ] I load onto an explicit device and use evaluation mode for deterministic inference.
- [ ] I verify fresh-object logits and predictions against a reference captured before saving.
- [ ] I know that exact training continuation may also require scheduler, scaler, sampler, and RNG states.